In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import preprocessing   
from sklearn.model_selection import train_test_split
from sklearn.cluster import k_means

In [20]:
df = pd.read_excel('online_retail.xlsx')
df.head()
pd.set_option('display.max_rows', None)

In [21]:
pd.unique(df['Country'])

array(['United Kingdom', 'France', 'Australia', 'Netherlands', 'Germany',
       'Norway', 'EIRE', 'Switzerland', 'Spain', 'Poland', 'Portugal',
       'Italy', 'Belgium', 'Lithuania', 'Japan', 'Iceland',
       'Channel Islands', 'Denmark', 'Cyprus', 'Sweden', 'Austria',
       'Israel', 'Finland', 'Bahrain', 'Greece', 'Hong Kong', 'Singapore',
       'Lebanon', 'United Arab Emirates', 'Saudi Arabia',
       'Czech Republic', 'Canada', 'Unspecified', 'Brazil', 'USA',
       'European Community', 'Malta', 'RSA'], dtype=object)

In [22]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [24]:
df.head(100)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [17]:
df = df.dropna(subset=['Description'],inplace=True)

In [18]:
df['Description'].value_counts()

TypeError: 'NoneType' object is not subscriptable

In [19]:
def categorize_products(description):
    description = str(description).lower()
    
    # Kitchen & Dining
    if any(word in description for word in ['lunch', 'bowl', 'plate', 'cup', 'mug', 'spoon', 'kitchen', 'tea', 'coffee', 
                                          'cake', 'baking', 'jar', 'tin', 'teapot', 'jug', 'pantry', 'oven', 'dining', 
                                          'cutlery', 'coaster', 'glass', 'bottle', 'fork', 'knife', 'saucer', 'food', 
                                          'drink', 'picnic', 'ceramic', 'recipe', 'cook', 'biscuit', 'sugar', 'milk',
                                          'set of', 'pack of', 'tray', 'coaster', 'dishes', 'pitcher', 'kettle',
                                          'cafetiere', 'toast', 'butter', 'egg cup', 'tablecloth']):
        return 'Kitchen & Dining'
    
    # Home Decor
    elif any(word in description for word in ['frame', 'picture', 'mirror', 'wall', 'clock', 'sign', 'decoration', 'ornament', 
                                            'lantern', 'hook', 'holder', 'shelf', 'cabinet', 'metal', 'wooden', 'zinc', 
                                            'hanging', 'stand', 'display', 'plaque', 'antique', 'vintage', 'retro', 'retrospot',
                                            'novelty', 'fun', 'funny', 'sign', 'plaque', 'home sweet home', 'chalkboard',
                                            'memo', 'notice', 'billboard', 'decorative', 'ornate', 'traditional']):
        return 'Home Decor'
    
    # Storage & Organization
    elif any(word in description for word in ['bag', 'shopper', 'rucksack', 'backpack', 'storage', 'box', 'container', 
                                            'basket', 'case', 'organizer', 'holder', 'rack', 'tidy', 'chest', 'drawer',
                                            'set', 'pack', 'gift box', 'jumbo bag', 'charlotte bag', 'storage', 'carrier']):
        return 'Storage & Organization'
    
    # Seasonal & Holiday
    elif any(word in description for word in ['christmas', 'xmas', 'santa', 'tree', 'snowman', 'reindeer', 'holly', 'festive', 
                                            'easter', 'halloween', 'bunting', 'decoration', 'party', 'valentine', 'wreath',
                                            'heart', 'love', 'romantic', 'gift set', 'hamper', 'present', 'festival',
                                            'celebration', 'holiday', 'seasonal', 'garland', 'bauble']):
        return 'Seasonal & Holiday'
    
    # Stationery & Gift Wrap
    elif any(word in description for word in ['pen', 'pencil', 'paper', 'card', 'notebook', 'craft', 'tape', 'wrap', 
                                            'gift tag', 'sticker', 'envelope', 'tissue', 'napkin', 'doily', 'doiley',
                                            'set of', 'pack of', 'writing', 'eraser', 'ruler', 'scissor', 'stamp']):
        return 'Stationery & Gift Wrap'
    
    # Children & Toys
    elif any(word in description for word in ['toy', 'game', 'play', 'doll', 'puzzle', 'children', 'kid', 'baby', 
                                            'spaceboy', 'dolly', 'charlie', 'lola', 'circus', 'woodland', 'gift set',
                                            'dinosaur', 'animal', 'fairy', 'magic', 'fun', 'puppet']):
        return 'Children & Toys'
    
    # Lighting & Candles
    elif any(word in description for word in ['candle', 't-light', 'tealight', 'light', 'lamp', 'chandelier', 'lighting', 
                                            'fairy', 'led', 'bulb', 'set of', 'lantern', 'torch', 'illuminat']):
        return 'Lighting & Candles'
    
    # Garden & Outdoor
    elif any(word in description for word in ['garden', 'flower', 'plant', 'watering', 'outdoor', 'bird', 'nest', 'pot', 
                                            'grow', 'seed', 'herb', 'floral', 'rose', 'artificial', 'set of', 'botanical',
                                            'planter', 'gardening', 'lawn', 'patio']):
        return 'Garden & Outdoor'
    
    # Fashion & Accessories
    elif any(word in description for word in ['umbrella', 'wallet', 'purse', 'handbag', 'scarf', 'hat', 'glove', 
                                            'cosmetic', 'key ring', 'keyring', 'glasses', 'phone', 'gift set',
                                            'jewel', 'necklace', 'bracelet', 'earring', 'crystal', 'pendant', 'charm']):
        return 'Fashion & Accessories'
    
    # Administrative
    elif any(word in description for word in ['damage', 'broken', 'missing', 'lost', 'wet', 'thrown', 'incorrect', 
                                            'adjust', 'display', 'sample', 'amazon', 'dotcom', 'check', 'return',
                                            'credit', 'wrong', 'error']):
        return 'Administrative'
    
    else:
        return 'Other'

# Create a new column with categorized products
df['Product_Category'] = df['Description'].apply(categorize_products)

# Display the distribution of the new categories
product_category_dist = df['Product_Category'].value_counts()
print("\nProduct Category Distribution:")
print(product_category_dist)

# Display the percentage distribution
percentage_dist = (product_category_dist / len(df)) * 100
print("\nPercentage Distribution:")
print(percentage_dist)

# Create a bar plot
plt.figure(figsize=(15, 8))
product_category_dist.plot(kind='bar')
plt.title('Distribution of Product Categories')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Check what's still in "Other"
other_items = df[df['Product_Category'] == 'Other']['Description'].value_counts().head(50)
print("\nTop 20 items still in 'Other' category:")
print(other_items)

TypeError: 'NoneType' object is not subscriptable